# 🏆 Ensemble Segmentation: DeepLabV3+ ⊕ SegFormer
## Road Marking Detection · Soft Voting + Hard Voting · Google Drive
> **Combines DeepLabV3+ (ResNet-101) and SegFormer (MiT-B2) predictions via probability averaging (soft vote) and majority vote (hard vote).**

## 📁 Cell 1 — Mount Google Drive

In [1]:
from google.colab import drive
drive.mount('/content/drive')
print('Drive mounted.')

Mounted at /content/drive
Drive mounted.


## 📦 Cell 2 — Install Dependencies

In [2]:
!pip install -q transformers>=4.37 accelerate timm
!pip install -q torch torchvision --index-url https://download.pytorch.org/whl/cu118
!pip install -q opencv-python-headless albumentations tqdm matplotlib scikit-learn
print('All dependencies installed.')

All dependencies installed.


## ⚙️ Cell 3 — Imports & Global Configuration

In [3]:
import os, glob, random, warnings, time
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import cv2
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from pathlib import Path
from tqdm.notebook import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torchvision.models as models

import albumentations as A
from albumentations.pytorch import ToTensorV2

from transformers import SegformerForSemanticSegmentation

# ── GLOBAL CONFIG ─────────────────────────────────────────────────────────
NUM_CLASSES  = 14          # 13 road markings + 1 background
IMG_SIZE     = 512
BATCH_SIZE   = 4           # Keep low for ensemble (two models in memory)
DEVICE       = 'cuda' if torch.cuda.is_available() else 'cpu'
MEAN         = [0.485, 0.456, 0.406]
STD          = [0.229, 0.224, 0.225]

# ── PATHS (EDIT THESE) ────────────────────────────────────────────────────
DATASET_ROOT       = '/content/drive/MyDrive/CVRoadMarkDetection/dataset'
TEST_IMG_DIR       = os.path.join(DATASET_ROOT, 'test', 'images')
TEST_LBL_DIR       = os.path.join(DATASET_ROOT, 'test', 'labels')

# Pre-computed mask cache dirs (same as individual training notebooks)
TEST_MASK_DIR      = '/content/drive/MyDrive/CVRoadMarkDetection/DeepLabV3+/masks/test'

# Checkpoint paths
DEEPLAB_CKPT       = '/content/drive/MyDrive/CVRoadMarkDetection/BestModels/best_finetune.pth'
SEGFORMER_CKPT     = '/content/drive/MyDrive/CVRoadMarkDetection/BestModels/segformer_best.pth'
SegFormer_id2label = '/content/drive/MyDrive/CVRoadMarkDetection/SegFormer/Training6/checkpoints/id2label.json'

# Output directory for ensemble results
OUTPUT_DIR = '/content/drive/MyDrive/CVRoadMarkDetection/Ensemble'
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ── CLASS METADATA ────────────────────────────────────────────────────────
CLASS_NAMES = [
    'Background', 'BUS LANE', 'Yellow Markings', 'Line 1', 'Line 2',
    'Crossing', 'Romb', 'SLOW', 'Left Arrow', 'Forward Arrow',
    'Fwd Arrow-Left', 'Fwd Arrow-Right', 'Right Arrow', 'Bicycle',
]
CLASS_COLORS = [
    (0,0,0),(0,0,255),(0,255,255),(0,255,0),(0,128,0),
    (255,255,0),(128,0,255),(255,0,0),(255,128,0),(255,0,255),
    (128,255,0),(0,128,255),(255,0,128),(128,128,0),
]

print(f'Device : {DEVICE}')
if DEVICE == 'cuda':
    print(f'GPU    : {torch.cuda.get_device_name(0)}')
print(f'Classes: {NUM_CLASSES}')
print(f'Output : {OUTPUT_DIR}')

Device : cuda
GPU    : Tesla T4
Classes: 14
Output : /content/drive/MyDrive/CVRoadMarkDetection/Ensemble


## 🏗️ Cell 4 — DeepLabV3+ Model Architecture (must match training)

In [4]:
class ASPPConv(nn.Sequential):
    def __init__(self, in_ch, out_ch, dilation):
        super().__init__(
            nn.Conv2d(in_ch, out_ch, 3, padding=dilation, dilation=dilation, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True)
        )

class ASPPPooling(nn.Sequential):
    def __init__(self, in_ch, out_ch):
        super().__init__(
            nn.AdaptiveAvgPool2d(1),
            nn.Conv2d(in_ch, out_ch, 1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True)
        )
    def forward(self, x):
        size = x.shape[-2:]
        for m in self: x = m(x)
        return F.interpolate(x, size=size, mode='bilinear', align_corners=False)

class ASPP(nn.Module):
    def __init__(self, in_ch=2048, out_ch=256, rates=(6, 12, 18)):
        super().__init__()
        mods = [nn.Sequential(
                    nn.Conv2d(in_ch, out_ch, 1, bias=False),
                    nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True))]
        for r in rates:
            mods.append(ASPPConv(in_ch, out_ch, r))
        mods.append(ASPPPooling(in_ch, out_ch))
        self.convs = nn.ModuleList(mods)
        self.project = nn.Sequential(
            nn.Conv2d(len(mods) * out_ch, out_ch, 1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            nn.Dropout(0.5)
        )
    def forward(self, x):
        return self.project(torch.cat([c(x) for c in self.convs], dim=1))

class DeepLabV3Plus(nn.Module):
    def __init__(self, num_classes=NUM_CLASSES, pretrained=False):
        super().__init__()
        bb = models.resnet101(weights='IMAGENET1K_V2' if pretrained else None)
        self.layer0 = nn.Sequential(bb.conv1, bb.bn1, bb.relu, bb.maxpool)
        self.layer1 = bb.layer1
        self.layer2 = bb.layer2
        self.layer3 = bb.layer3
        self.layer4 = bb.layer4
        for n, m in self.layer3.named_modules():
            if 'conv2' in n: m.dilation=(2,2); m.padding=(2,2)
        for n, m in self.layer4.named_modules():
            if 'conv2' in n: m.dilation=(4,4); m.padding=(4,4)
        self.aspp     = ASPP(2048, 256, (6, 12, 18))
        self.low_proj = nn.Sequential(
            nn.Conv2d(256, 48, 1, bias=False), nn.BatchNorm2d(48), nn.ReLU(inplace=True))
        self.decoder  = nn.Sequential(
            nn.Conv2d(304, 256, 3, padding=1, bias=False),
            nn.BatchNorm2d(256), nn.ReLU(inplace=True), nn.Dropout(0.5),
            nn.Conv2d(256, 256, 3, padding=1, bias=False),
            nn.BatchNorm2d(256), nn.ReLU(inplace=True), nn.Dropout(0.1),
            nn.Conv2d(256, num_classes, 1)
        )
    def forward(self, x):
        in_size = x.shape[-2:]
        x = self.layer0(x)
        low = self.layer1(x)
        x = self.layer2(low)
        x = self.layer3(x)
        x = self.layer4(x)
        x = self.aspp(x)
        x = F.interpolate(x, size=low.shape[-2:], mode='bilinear', align_corners=False)
        low = self.low_proj(low)
        x = self.decoder(torch.cat([x, low], dim=1))
        return F.interpolate(x, size=in_size, mode='bilinear', align_corners=False)

print('DeepLabV3+ architecture defined.')

DeepLabV3+ architecture defined.


## 🔄 Cell 5 — Load Both Model Weights from Drive

In [5]:
def load_deeplab(ckpt_path: str, device: str) -> nn.Module:
    """Instantiate DeepLabV3+ and load fine-tuned weights."""
    model = DeepLabV3Plus(num_classes=NUM_CLASSES, pretrained=False).to(device)
    ckpt  = torch.load(ckpt_path, map_location=device, weights_only=False)
    state = ckpt.get('model_state_dict', ckpt)  # support both formats
    model.load_state_dict(state)
    model.eval()
    print(f'[DeepLab]  Loaded  {ckpt_path}')
    print(f'           Params  {sum(p.numel() for p in model.parameters())/1e6:.1f}M')
    if 'val_miou' in ckpt:
        print(f'           Val mIoU (ckpt): {ckpt["val_miou"]:.4f}')
    return model


def load_segformer(ckpt_path: str, device: str) -> nn.Module:
    """Instantiate SegFormer MiT-B2 and load fine-tuned weights."""
    id2label = {0: 'background', **{i+1: CLASS_NAMES[i+1] for i in range(NUM_CLASSES-1)}}
    label2id = {v: k for k, v in id2label.items()}
    model = SegformerForSemanticSegmentation.from_pretrained(
        'nvidia/mit-b2',
        num_labels=NUM_CLASSES,
        id2label=id2label,
        label2id=label2id,
        ignore_mismatched_sizes=True
    ).to(device)
    ckpt  = torch.load(ckpt_path, map_location=device, weights_only=False)
    state = ckpt.get('model_state_dict', ckpt)
    model.load_state_dict(state)
    model.eval()
    print(f'[SegFormer] Loaded  {ckpt_path}')
    print(f'            Params  {sum(p.numel() for p in model.parameters())/1e6:.1f}M')
    if 'best_miou' in ckpt:
        print(f'            Best mIoU (ckpt): {ckpt["best_miou"]:.4f}')
    return model


deeplab_model   = load_deeplab(DEEPLAB_CKPT,   DEVICE)
segformer_model = load_segformer(SEGFORMER_CKPT, DEVICE)
print('\n✅ Both models loaded and in eval mode.')

RuntimeError: Error(s) in loading state_dict for DeepLabV3Plus:
	Missing key(s) in state_dict: "aspp.convs.4.2.weight", "aspp.convs.4.2.bias", "aspp.convs.4.2.running_mean", "aspp.convs.4.2.running_var", "decoder.4.weight", "decoder.5.bias", "decoder.5.running_mean", "decoder.5.running_var", "decoder.8.weight", "decoder.8.bias". 
	Unexpected key(s) in state_dict: "aspp.convs.5.1.weight", "aspp.convs.5.2.weight", "aspp.convs.5.2.bias", "aspp.convs.5.2.running_mean", "aspp.convs.5.2.running_var", "aspp.convs.5.2.num_batches_tracked", "aspp.convs.4.0.weight", "aspp.convs.4.1.bias", "aspp.convs.4.1.running_mean", "aspp.convs.4.1.running_var", "aspp.convs.4.1.num_batches_tracked", "decoder.9.weight", "decoder.9.bias", "decoder.3.se.2.weight", "decoder.3.se.4.weight", "decoder.6.weight", "decoder.6.bias", "decoder.6.running_mean", "decoder.6.running_var", "decoder.6.num_batches_tracked". 
	size mismatch for aspp.convs.4.1.weight: copying a param with shape torch.Size([256]) from checkpoint, 

## 📂 Cell 6 — Dataset & DataLoader

In [ ]:
class RoadMarkingDataset(Dataset):
    """
    Shared dataset for both models.
    Returns: (image_tensor, mask_tensor, img_path_str)
    """
    def __init__(self, img_dir, mask_dir, transform=None):
        self.mask_dir  = mask_dir
        self.transform = transform
        all_imgs = sorted(
            glob.glob(os.path.join(img_dir, '*.jpg')) +
            glob.glob(os.path.join(img_dir, '*.png'))
        )
        self.samples = [
            p for p in all_imgs
            if os.path.exists(os.path.join(mask_dir, Path(p).stem + '.png'))
        ]
        print(f'  [{os.path.basename(img_dir)}] {len(self.samples)} labelled samples found.')

    def __len__(self): return len(self.samples)

    def __getitem__(self, idx):
        ip   = self.samples[idx]
        mp   = os.path.join(self.mask_dir, Path(ip).stem + '.png')
        img  = cv2.cvtColor(cv2.imread(ip), cv2.COLOR_BGR2RGB)
        img  = cv2.resize(img, (IMG_SIZE, IMG_SIZE))
        mask = cv2.imread(mp, cv2.IMREAD_GRAYSCALE)
        mask = cv2.resize(mask, (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_NEAREST)
        if self.transform:
            aug  = self.transform(image=img, mask=mask)
            img, mask = aug['image'], aug['mask']
        return img, mask.long(), ip


eval_tfm = A.Compose([
    A.Resize(IMG_SIZE, IMG_SIZE),
    A.Normalize(mean=MEAN, std=STD),
    ToTensorV2(),
])

test_ds     = RoadMarkingDataset(TEST_IMG_DIR, TEST_MASK_DIR, transform=eval_tfm)
test_loader = DataLoader(
    test_ds, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=2, pin_memory=(DEVICE == 'cuda')
)
print(f'Test batches: {len(test_loader)}')

## 🧩 Cell 7 — Ensemble Core: Soft Vote + Hard Vote

| Method | Description |
|--------|-------------|
| **Soft Vote** | Average softmax probabilities → argmax. Best when models are calibrated. |
| **Hard Vote** | Each model casts a class vote per-pixel → majority wins. |
| **Weighted Soft** | Like soft vote but each model's probs are scaled by a learnable weight. |

In [ ]:
class EnsemblePredictor:
    """
    Fuses DeepLabV3+ and SegFormer predictions.

    Args:
        models       : list of nn.Module (already on device, eval mode)
        weights      : per-model probability weights (default equal)
        mode         : 'soft'  – average probabilities (recommended)
                       'hard'  – per-pixel majority vote
                       'weighted_soft' – weighted probability average
        tta          : enable Test-Time Augmentation (h-flip)
    """

    def __init__(self, models, weights=None, mode='soft', tta=False):
        self.models  = models
        self.mode    = mode
        self.tta     = tta
        n = len(models)
        if weights is None:
            self.weights = [1.0 / n] * n
        else:
            s = sum(weights)
            self.weights = [w / s for w in weights]

    @torch.no_grad()
    def _forward_single(self, model, imgs):
        """Run one model; handle DeepLab (direct logits) vs SegFormer (HF output)."""
        try:
            # SegFormer returns an object with .logits
            out = model(pixel_values=imgs)
            logits = out.logits  # [B, C, H/4, W/4]
        except TypeError:
            logits = model(imgs)  # DeepLabV3+ returns tensor directly
        # Upsample to input spatial size
        return F.interpolate(
            logits, size=(IMG_SIZE, IMG_SIZE),
            mode='bilinear', align_corners=False
        )  # [B, C, H, W]

    @torch.no_grad()
    def _get_probs(self, model, imgs):
        """Get softmax probs, optionally with h-flip TTA."""
        logits = self._forward_single(model, imgs)
        probs  = F.softmax(logits, dim=1)
        if self.tta:
            imgs_flip   = torch.flip(imgs, dims=[-1])
            logits_flip = self._forward_single(model, imgs_flip)
            probs_flip  = F.softmax(logits_flip, dim=1)
            probs_flip  = torch.flip(probs_flip, dims=[-1])  # un-flip
            probs       = (probs + probs_flip) * 0.5
        return probs  # [B, C, H, W]

    @torch.no_grad()
    def predict(self, imgs):
        """
        Returns:
            preds_soft : [B, H, W] int64  – soft-vote prediction
            preds_hard : [B, H, W] int64  – hard-vote prediction
            avg_probs  : [B, C, H, W] float – averaged probabilities
        """
        all_probs  = []
        all_argmax = []

        for model, w in zip(self.models, self.weights):
            probs = self._get_probs(model, imgs)     # [B, C, H, W]
            all_probs.append(probs * w)
            all_argmax.append(probs.argmax(dim=1))  # [B, H, W]

        # ── Soft Vote ────────────────────────────────────────────────────
        avg_probs  = torch.stack(all_probs, dim=0).sum(dim=0)  # [B, C, H, W]
        preds_soft = avg_probs.argmax(dim=1)                   # [B, H, W]

        # ── Hard Vote ────────────────────────────────────────────────────
        votes      = torch.stack(all_argmax, dim=0)            # [M, B, H, W]
        # For each pixel, pick the class with most votes
        # Fast vectorised majority vote via one-hot accumulation
        B, H, W    = votes.shape[1], votes.shape[2], votes.shape[3]
        vote_acc   = torch.zeros(B, NUM_CLASSES, H, W, device=imgs.device)
        for v in all_argmax:
            vote_acc.scatter_add_(
                1,
                v.unsqueeze(1),
                torch.ones(B, 1, H, W, device=imgs.device)
            )
        preds_hard = vote_acc.argmax(dim=1)                    # [B, H, W]

        return preds_soft, preds_hard, avg_probs


# Instantiate with equal weights and TTA enabled
# Adjust weights if one model is known to be stronger (e.g. [0.55, 0.45])
ensemble = EnsemblePredictor(
    models  = [deeplab_model, segformer_model],
    weights = [0.5, 0.5],
    mode    = 'soft',
    tta     = True          # flip augmentation at test time
)
print('✅ EnsemblePredictor ready  (soft vote + hard vote, TTA=True)')

## 📐 Cell 8 — Evaluation Utilities (mIoU, per-class IoU)

In [ ]:
def compute_iou_matrix(preds: torch.Tensor, targets: torch.Tensor,
                        n_cls: int = NUM_CLASSES):
    """
    Returns:
        per_class_iou : np.ndarray [n_cls]  (NaN for absent classes)
        miou          : float
    """
    preds   = preds.cpu().numpy().ravel()
    targets = targets.cpu().numpy().ravel()
    ious    = np.full(n_cls, np.nan)
    for c in range(n_cls):
        tp    = np.sum((preds == c) & (targets == c))
        fp    = np.sum((preds == c) & (targets != c))
        fn    = np.sum((preds != c) & (targets == c))
        denom = tp + fp + fn
        if denom > 0:
            ious[c] = tp / denom
    miou = float(np.nanmean(ious[1:]))  # skip background
    return ious, miou


def aggregate_ious(all_ious: list):
    """Macro-average IoU across batches (ignore NaN = absent class)."""
    stacked = np.vstack(all_ious)  # [N_batches, n_cls]
    mean    = np.nanmean(stacked, axis=0)
    miou    = float(np.nanmean(mean[1:]))
    return mean, miou


def colorise_mask(mask: np.ndarray) -> np.ndarray:
    colour = np.zeros((*mask.shape, 3), dtype=np.uint8)
    for cls_id, rgb in enumerate(CLASS_COLORS):
        colour[mask == cls_id] = rgb
    return colour

print('Evaluation utilities defined.')

## 🔬 Cell 9 — Full Evaluation on Test Set

In [ ]:
@torch.no_grad()
def evaluate_ensemble(ensemble, loader, device=DEVICE):
    """
    Runs the ensemble over the full test set.
    Returns a dict with mIoU for each strategy + individual models.
    """
    ious_soft     = []
    ious_hard     = []
    ious_deeplab  = []
    ious_segformer= []

    pbar = tqdm(loader, desc='Evaluating ensemble', leave=True)
    for imgs, masks, _ in pbar:
        imgs  = imgs.to(device)
        masks = masks.to(device)

        # ── Ensemble predictions ────────────────────────────────────────
        p_soft, p_hard, avg_probs = ensemble.predict(imgs)

        # ── Individual model predictions (for comparison) ───────────────
        dl_logits = ensemble._forward_single(deeplab_model, imgs)
        sf_logits = ensemble._forward_single(segformer_model, imgs)
        p_deeplab   = dl_logits.argmax(1)
        p_segformer = sf_logits.argmax(1)

        for b in range(imgs.size(0)):
            gt = masks[b]
            _, miou_s  = compute_iou_matrix(p_soft[b],     gt)
            _, miou_h  = compute_iou_matrix(p_hard[b],     gt)
            _, miou_dl = compute_iou_matrix(p_deeplab[b],  gt)
            _, miou_sf = compute_iou_matrix(p_segformer[b],gt)

            iou_s, _  = compute_iou_matrix(p_soft[b],     gt)
            iou_h, _  = compute_iou_matrix(p_hard[b],     gt)
            iou_dl, _ = compute_iou_matrix(p_deeplab[b],  gt)
            iou_sf, _ = compute_iou_matrix(p_segformer[b],gt)

            ious_soft.append(iou_s)
            ious_hard.append(iou_h)
            ious_deeplab.append(iou_dl)
            ious_segformer.append(iou_sf)

        pbar.set_postfix(soft=f'{miou_s:.3f}', hard=f'{miou_h:.3f}')

    pc_soft,     miou_soft     = aggregate_ious(ious_soft)
    pc_hard,     miou_hard     = aggregate_ious(ious_hard)
    pc_deeplab,  miou_deeplab  = aggregate_ious(ious_deeplab)
    pc_segformer,miou_segformer= aggregate_ious(ious_segformer)

    results = {
        'soft_vote'    : {'miou': miou_soft,      'per_class': pc_soft},
        'hard_vote'    : {'miou': miou_hard,      'per_class': pc_hard},
        'deeplab_only' : {'miou': miou_deeplab,   'per_class': pc_deeplab},
        'segformer_only':{'miou': miou_segformer, 'per_class': pc_segformer},
    }

    print('\n' + '='*60)
    print(f'  DeepLabV3+  only   mIoU = {miou_deeplab:.4f}')
    print(f'  SegFormer   only   mIoU = {miou_segformer:.4f}')
    print(f'  Soft Vote ensemble mIoU = {miou_soft:.4f}  ← recommended')
    print(f'  Hard Vote ensemble mIoU = {miou_hard:.4f}')
    print('='*60)
    return results


results = evaluate_ensemble(ensemble, test_loader)


## 📊 Cell 10 — Per-Class IoU Comparison Chart

In [ ]:
def plot_perclass_comparison(results, save_path=None):
    labels  = CLASS_NAMES[1:]   # skip background
    x       = np.arange(len(labels))
    w       = 0.2
    colors  = ['steelblue', 'darkorange', 'forestgreen', 'crimson']
    keys    = ['deeplab_only', 'segformer_only', 'soft_vote', 'hard_vote']
    descs   = ['DeepLab', 'SegFormer', 'Soft Vote', 'Hard Vote']

    fig, ax = plt.subplots(figsize=(18, 6))
    for i, (k, d, c) in enumerate(zip(keys, descs, colors)):
        iou_vals = results[k]['per_class'][1:]  # skip background
        ax.bar(x + (i - 1.5) * w, iou_vals, w,
               label=f'{d} ({results[k]["miou"]:.3f})', color=c, alpha=0.85)

    ax.set_xticks(x)
    ax.set_xticklabels(labels, rotation=35, ha='right', fontsize=8)
    ax.set_ylabel('IoU'); ax.set_ylim(0, 1.05)
    ax.set_title('Per-Class IoU: Individual Models vs Ensemble Strategies', fontsize=13)
    ax.legend(loc='upper right', fontsize=9)
    ax.grid(axis='y', alpha=0.35)
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
        print(f'Saved: {save_path}')
    plt.show()

plot_perclass_comparison(
    results,
    save_path=os.path.join(OUTPUT_DIR, 'perclass_comparison.png')
)

## 📋 Cell 11 — Per-Class IoU Table & CSV Export

In [ ]:
rows = []
for cls_idx, name in enumerate(CLASS_NAMES):
    rows.append({
        'class'         : name,
        'deeplab_iou'   : round(float(results['deeplab_only']['per_class'][cls_idx] or 0), 4),
        'segformer_iou' : round(float(results['segformer_only']['per_class'][cls_idx] or 0), 4),
        'soft_vote_iou' : round(float(results['soft_vote']['per_class'][cls_idx] or 0), 4),
        'hard_vote_iou' : round(float(results['hard_vote']['per_class'][cls_idx] or 0), 4),
    })

df = pd.DataFrame(rows)

# Append summary row
summary = pd.DataFrame([{
    'class'         : 'mIoU (excl. BG)',
    'deeplab_iou'   : round(results['deeplab_only']['miou'], 4),
    'segformer_iou' : round(results['segformer_only']['miou'], 4),
    'soft_vote_iou' : round(results['soft_vote']['miou'], 4),
    'hard_vote_iou' : round(results['hard_vote']['miou'], 4),
}])
df = pd.concat([df, summary], ignore_index=True)

csv_path = os.path.join(OUTPUT_DIR, 'ensemble_iou_results.csv')
df.to_csv(csv_path, index=False)
print(f'Results saved to: {csv_path}')
df

## 🖼️ Cell 12 — Visual Inference: Side-by-Side Comparison

In [ ]:
@torch.no_grad()
def visualize_ensemble(ensemble, img_paths, n=5, save_dir=OUTPUT_DIR):
    """
    For each sample: Image | GT | DeepLab | SegFormer | Soft Vote | Hard Vote
    """
    chosen = random.sample(img_paths, min(n, len(img_paths)))
    tfm    = A.Compose([A.Resize(IMG_SIZE, IMG_SIZE),
                        A.Normalize(mean=MEAN, std=STD), ToTensorV2()])
    cols   = ['Image', 'Ground Truth', 'DeepLab', 'SegFormer', 'Soft Vote', 'Hard Vote']

    fig, axes = plt.subplots(n, 6, figsize=(24, 4 * n))
    if n == 1: axes = [axes]

    for i, ip in enumerate(chosen):
        stem = Path(ip).stem
        raw  = cv2.cvtColor(cv2.imread(ip), cv2.COLOR_BGR2RGB)
        raw  = cv2.resize(raw, (IMG_SIZE, IMG_SIZE))

        # Ground truth mask
        mp   = os.path.join(TEST_MASK_DIR, stem + '.png')
        gt   = cv2.imread(mp, cv2.IMREAD_GRAYSCALE)
        if gt is not None:
            gt = cv2.resize(gt, (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_NEAREST)

        t    = tfm(image=raw)['image'].unsqueeze(0).to(DEVICE)

        # Predictions
        p_soft, p_hard, _ = ensemble.predict(t)
        dl_log = ensemble._forward_single(deeplab_model, t)
        sf_log = ensemble._forward_single(segformer_model, t)
        p_dl   = dl_log.argmax(1).squeeze().cpu().numpy().astype(np.uint8)
        p_sf   = sf_log.argmax(1).squeeze().cpu().numpy().astype(np.uint8)
        p_sv   = p_soft.squeeze().cpu().numpy().astype(np.uint8)
        p_hv   = p_hard.squeeze().cpu().numpy().astype(np.uint8)

        frames = [raw,
                  colorise_mask(gt) if gt is not None else np.zeros((IMG_SIZE,IMG_SIZE,3),np.uint8),
                  colorise_mask(p_dl), colorise_mask(p_sf),
                  colorise_mask(p_sv), colorise_mask(p_hv)]

        for j, (frame, col) in enumerate(zip(frames, cols)):
            axes[i][j].imshow(frame)
            axes[i][j].set_title(col, fontsize=8)
            axes[i][j].axis('off')

    patches = [mpatches.Patch(color=np.array(c)/255, label=n_)
               for c, n_ in zip(CLASS_COLORS, CLASS_NAMES)]
    fig.legend(handles=patches, loc='lower center', ncol=7, fontsize=7,
               bbox_to_anchor=(0.5, -0.02))
    plt.suptitle('Ensemble Comparison: DeepLabV3+ ⊕ SegFormer', fontsize=14, fontweight='bold')
    plt.tight_layout()
    out_path = os.path.join(save_dir, 'ensemble_visual.png')
    plt.savefig(out_path, dpi=120, bbox_inches='tight')
    print(f'Saved: {out_path}')
    plt.show()


test_imgs = sorted(
    glob.glob(os.path.join(TEST_IMG_DIR, '*.jpg')) +
    glob.glob(os.path.join(TEST_IMG_DIR, '*.png'))
)
visualize_ensemble(ensemble, test_imgs, n=5)

## 🎥 Cell 13 — Ensemble Video Inference

In [ ]:
@torch.no_grad()
def process_video_ensemble(ensemble, input_path, output_path, device=DEVICE):
    """
    Run soft-vote ensemble on each video frame and write overlay video.
    """
    cap    = cv2.VideoCapture(input_path)
    W      = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    H      = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    fps    = cap.get(cv2.CAP_PROP_FPS)
    total  = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    out    = cv2.VideoWriter(output_path, fourcc, fps, (W, H))

    tfm = A.Compose([
        A.Resize(IMG_SIZE, IMG_SIZE),
        A.Normalize(mean=MEAN, std=STD),
        ToTensorV2(),
    ])

    for _ in tqdm(range(total), desc='Ensemble video'):
        ret, frame = cap.read()
        if not ret: break

        rgb    = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        tensor = tfm(image=rgb)['image'].unsqueeze(0).to(device)
        p_soft, _, _ = ensemble.predict(tensor)  # soft vote
        mask   = p_soft.squeeze().cpu().numpy().astype(np.uint8)

        col_mask = colorise_mask(mask)
        col_mask = cv2.resize(col_mask, (W, H), interpolation=cv2.INTER_NEAREST)
        mask_bgr = cv2.cvtColor(col_mask, cv2.COLOR_RGB2BGR)
        overlay  = cv2.addWeighted(frame, 0.6, mask_bgr, 0.4, 0)
        out.write(overlay)

    cap.release(); out.release()
    print(f'Video saved: {output_path}')


# ── USAGE: edit paths and uncomment ───────────────────────────────────────
# INPUT_VID  = '/content/drive/MyDrive/CVRoadMarkDetection/videos/3.webm'
# OUTPUT_VID = os.path.join(OUTPUT_DIR, 'ensemble_3.mp4')
# process_video_ensemble(ensemble, INPUT_VID, OUTPUT_VID)
print('Video inference function defined. Uncomment above to run.')

## 💾 Cell 14 — Save Ensemble Config & Results Summary

In [ ]:
import json, datetime

config = {
    'created'         : datetime.datetime.now().isoformat(),
    'models'          : [
        {'name': 'DeepLabV3+', 'backbone': 'ResNet-101', 'ckpt': DEEPLAB_CKPT},
        {'name': 'SegFormer',  'backbone': 'MiT-B2',     'ckpt': SEGFORMER_CKPT},
    ],
    'ensemble_weights': ensemble.weights,
    'tta'             : ensemble.tta,
    'num_classes'     : NUM_CLASSES,
    'img_size'        : IMG_SIZE,
    'results'         : {
        k: {'miou': round(v['miou'], 4),
            'per_class': {CLASS_NAMES[i]: round(float(v['per_class'][i] or 0), 4)
                          for i in range(NUM_CLASSES)}}
        for k, v in results.items()
    }
}

cfg_path = os.path.join(OUTPUT_DIR, 'ensemble_config.json')
with open(cfg_path, 'w') as f:
    json.dump(config, f, indent=2)

print(f'Config saved: {cfg_path}')
print('\n── FINAL RESULTS SUMMARY ──────────────────────────────')
for k, v in results.items():
    print(f'  {k:<22} mIoU = {v["miou"]:.4f}')
print('─────────────────────────────────────────────────────')
print(f'  Output directory: {OUTPUT_DIR}')